# **1. Seguranca: Escondendo chave api**
So para mostrar, ja que nao vai ser usada neste desafio

In [ ]:
from google.colab import userdata
api_key = userdata.get('MINHA_API_KEY')

SecretNotFoundError: Secret MINHA_API_KEY does not exist.

In [ ]:

import pdfplumber
import pandas as pd
import gdown

# **2. Baixando o arquivo direto do link**

In [ ]:
# 1. Download automatizado do arquivo utilizando o ID do Google Drive
file_id = '1YKdnI4ziFCq9hoeXqTaje3BIL7BVwyI4'
url_download = f'https://drive.google.com/uc?id={file_id}'
arquivo_destino = 'remume_versao_2024.pdf'
gdown.download(url_download, arquivo_destino, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1YKdnI4ziFCq9hoeXqTaje3BIL7BVwyI4
To: /content/remume_versao_2024.pdf
100%|██████████| 1.05M/1.05M [00:00<00:00, 30.0MB/s]


'remume_versao_2024.pdf'

# **3.  Extração de Dados de PDFs**

## **Extracao de Texto**

In [ ]:
texto_total = ""

for pagina in pdf.pages:
    texto = pagina.extract_text()

    if texto:
        texto_total += texto + "\n"

print(texto_total[:500])

GOVERNO DE CRICIÚMA
PODER EXECUTIVO
SECRETARIA MUNICIPAL DE SAÚDE
RELAÇÃO MUNICIPAL DE MEDICAMENTOS ESSENCIAIS (REMUME)
LISTA DE MEDICAMENTOS DISPONIBILIZADOS
PELA REDE MUNICIPAL DE SAÚDE DE CRICIÚMA
Versão de atualização: novembro/2024
Prefeitura Municipal de Criciúma - Secretaria de Saúde - Assistência Farmacêutica - Relação Municipal de
Medicamentos - REMUME
REMUME CRICIÚMA - Versão 11/2024 - Página 2 de 27
Prefeitura Municipal de Criciúma - Secretaria de Saúde - Assistência Farmacêutica - Re


## **Extracao de tabela e tratamento dela**

In [ ]:
tabelas_totais = []

for pagina in pdf.pages:

    tabelas = pagina.extract_tables()

    for tabela in tabelas:

        for linha in tabela:
            tabelas_totais.append(linha)

In [ ]:
dados_limpos = []

for linha in tabelas_totais:

    if linha is not None:

        # remove linhas vazias
        linha = [str(item).strip() if item else "" for item in linha]

        dados_limpos.append(linha)

In [ ]:
max_colunas = max(len(linha) for linha in dados_limpos)

print(max_colunas)

7


In [ ]:
colunas = [f"coluna_{i}" for i in range(max_colunas)]

In [ ]:
dados_padronizados = []

for linha in dados_limpos:

    while len(linha) < max_colunas:
        linha.append("")

    dados_padronizados.append(linha)

In [ ]:
df = pd.DataFrame(dados_padronizados, columns=colunas)

df.head()

,coluna_0,coluna_1,coluna_2,coluna_3,coluna_4,coluna_5,coluna_6
0,Item,Medicamento,Apresentação,Local de acesso,,Classificação e tipo de\nreceita,
1,01,Acetilcisteína,600 mg\ngranulado\nenvelopes com\n5 g,- Farmácias Distritais,,- Expectorante\n- Receituário simples,
2,02,Aciclovir,200 mg\nComprimido,- Farmácias Distritais\n- Farmácia Estratégica,,- Antiviral\n- Receituário simples,
3,03,Aciclovir,50 mg/g\nCreme bisnaga,,- Farmácias Distritais,- Antiviral\n- Receituário simples,
4,,,,,- Farmácia Estratégica,,


In [ ]:
df.to_csv("remume_extraido.csv", index=False)

## **Aqui formatei ele de acordo com as colunas do pdf**
A extracao da tabela esta vindo errado

In [ ]:
# Extração forçando o alinhamento das 5 colunas reais

tabelas_totais = []

# Configuração estratégica
ajustes_tabela = {
    "vertical_strategy": "lines",
    "horizontal_strategy": "lines",
    "snap_tolerance": 5,
    "join_tolerance": 5,
}

for pagina in pdf.pages:

    tabelas = pagina.extract_tables(table_settings=ajustes_tabela)

    for tabela in tabelas:
        for linha in tabela:

            if any(linha):
                tabelas_totais.append(linha)

print(f"Total de linhas na memória: {len(tabelas_totais)}")

Total de linhas na memória: 289


In [ ]:
# Gerando o DataFrame com as 5 colunas exatas do PDF

nomes_colunas = ["Item", "Medicamento", "Apresentação", "Local de acesso", "Classificação e tipo de receita"]


if tabelas_totais[0][0] == "Item":
    dados_finais = tabelas_totais[1:]
else:
    dados_finais = tabelas_totais

df = pd.DataFrame(dados_finais)


df = df.iloc[:, :5]
df.columns = nomes_colunas

df = df[df["Item"] != "Item"].dropna(subset=["Medicamento"])

display(df.head(10))

,Item,Medicamento,Apresentação,Local de acesso,Classificação e tipo de receita
0,01,Acetilcisteína,600 mg\ngranulado\nenvelopes com\n5 g,- Farmácias Distritais,- Expectorante\n- Receituário simples
1,02,Aciclovir,200 mg\nComprimido,- Farmácias Distritais\n- Farmácia Estratégica,- Antiviral\n- Receituário simples
2,03,Aciclovir,50 mg/g\nCreme bisnaga,- Farmácias Distritais\n- Farmácia Estratégica...,- Antiviral\n- Receituário simples
3,04,Ácido acetilsalicílico,100 mg\nComprimido,- Farmácias Distritais\n- Dispensários nas UBS,- Antitrombótico\n- Receituário simples
4,05,Ácido fólico,5 mg\nComprimido,- Farmácias Distritais\n- Farmácia Estratégica...,- Vitamina\n- Receituário simples
5,06,Ácido fólico,"0,2 mg/ml\nSolução oral\nFrasco",- Farmácias Distritais\n- Dispensários nas UBS,- Vitamina\n- Receituário simples
6,07,Ácido valpróico\n(valproato de sódio)\n(Contro...,"50 mg/ml\n(equivalente a\n57,624mg ácido\nvalp...",- Farmácias Distritais,- Anticonvulsivante\n- Receituário de Controle...
7,08,Ácido valpróico\n(valproato de sódio)\n(Contro...,500 mg\n(equivalente a\n576 mg ácido\nvalpróic...,- Farmácias Distritais,- Anticonvulsivante\n- Receituário de Controle...
8,09,Ácido valpróico\n(valproato de sódio),250 mg\n(equivalente a,- Farmácias Distritais,- Anticonvulsivante\n- Receituário de Controle...
9,,(Controlado Port.\n344/98 - Lista C1),288 mg\nvalproato de\nsódio)\nCápsula,,
